In [ ]:
# STEP 1 — Data extraction

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_excel("../data/Ecommerce_Sales_Data.csv.xlsx")

# Display first 5 rows
print(df.head())

# Display dataset shape
print("Dataset Shape:", df.shape)

# Display column names
print("\nColumns:")
##print(df.columns.tolist())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
Dataset Shape: (541909, 8)

Columns:


In [3]:
#  STEP 2 — Initial data inspection

# Dataset information
print("\nDataset Information:")
df.info()

# Statistical summary
print("\nStatistical Summary:")
print(df.describe())

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())


Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB

Statistical Summary:
            Quantity                 InvoiceDate      UnitPrice     CustomerID
count  541909.000000                      541909  541909.000000  406829.000000
mean        9.552250  2011-07-04 13:34:57.156386       4.611114   15287.690570
min    -80995.000000         2

In [4]:
# STEP 3 — Duplicate removal

duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 5268


In [6]:
# STEP 4 — Data type correction

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

In [7]:
print(df["InvoiceDate"].dtype)

datetime64[us]


In [8]:
# STEP 5 — Detect cancelled transactions

cancelled = df[
    df["InvoiceNo"].astype(str).str.startswith("C")
]

print("Cancelled transactions:", len(cancelled))

Cancelled transactions: 9288


In [9]:
# remove for normal sales analysis

df = df[
    ~df["InvoiceNo"].astype(str).str.startswith("C")
].copy()

In [10]:
# STEP 6 — Invalid quantity values

print("Negative Quantity:")
print((df["Quantity"] < 0).sum())

Negative Quantity:
1336


In [11]:
df = df[df["Quantity"] > 0].copy()

In [12]:
# STEP 7 — Invalid price values

print("Zero or Negative Unit Price:")
print((df["UnitPrice"] <= 0).sum())

Zero or Negative Unit Price:
1181


In [14]:
# STEP 8 — Create an important business variable

df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

In [15]:
print(df[[
    "Quantity",
    "UnitPrice",
    "TotalAmount"
]].head())

   Quantity  UnitPrice  TotalAmount
0         6       2.55        15.30
1         6       3.39        20.34
2         8       2.75        22.00
3         6       3.39        20.34
4         6       3.39        20.34


In [16]:
# STEP 9 — Create time-based variables

df["Year"] = df["InvoiceDate"].dt.year

df["Month"] = df["InvoiceDate"].dt.month

df["MonthName"] = df["InvoiceDate"].dt.month_name()

df["Day"] = df["InvoiceDate"].dt.day

df["Weekday"] = df["InvoiceDate"].dt.day_name()

In [17]:
# STEP 10 — Outlier detection

Q1 = df["Quantity"].quantile(0.25)

Q3 = df["Quantity"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR

upper_bound = Q3 + 1.5 * IQR

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Lower Bound: -12.5
Upper Bound: 23.5


In [18]:
# STEP 11 — Normalization

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["Quantity_Normalized"] = scaler.fit_transform(
    df[["Quantity"]]
)

In [19]:
# STEP 12 — Final cleaned dataset

print("Final Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Final Shape: (531285, 15)

Missing Values:
InvoiceNo                   0
StockCode                   0
Description               592
Quantity                    0
InvoiceDate                 0
UnitPrice                   0
CustomerID             133361
Country                     0
TotalAmount                 0
Year                        0
Month                       0
MonthName                   0
Day                         0
Weekday                     0
Quantity_Normalized         0
dtype: int64

Duplicate Rows:
5231
